In [13]:
%load_ext autoreload
%autoreload 2
import numpy as np
from matplotlib import pyplot as plt
from astropy import units as u
from astropy import constants as const
from sklearn.metrics import r2_score
import pickle

from functions.load_data_and_var import load_data_and_var, generate_nfws_from_combined_dataset_werr
from functions.plotting import savefig, paper_plot
from functions.statistics_functions import chi_squared

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
data, var = load_data_and_var()

In [3]:
nfws_full=generate_nfws_from_combined_dataset_werr(data, var)

In [4]:
len(nfws_full)

173

In [5]:
nfws = [nfw for nfw in nfws_full if nfw.z < 0.2]
len(nfws)

173

In [9]:
nfws_subset = [nfw for nfw in nfws if nfw.energies.is_virialized_sp ]
zs = [nfw.z for nfw in nfws_subset]
filtered_vir = [nfw.energies.virialization_fraction_surface_pressure for nfw in nfws_subset]

len(nfws_subset)

51

In [10]:
#heating_rates = np.array([nfw.total_heating_rate(0.001*nfw.R500, nfw.R500, 0.1*nfw.R500) for nfw in nfws_subset])

In [11]:
#cooling_rates = np.array([nfw.total_cooling_rate(0.001*nfw.R500, nfw.R500, vol_rate='gfe') for nfw in nfws_subset])

In [12]:
equilibrium_clusters = []
eq_heating_rates = []
eq_cooling_rates = []
for n in nfws_subset:
    h = n.total_heating_rate(0.001*n.R500, n.R500, 0.1*n.R500)
    c = n.total_cooling_rate(0.001*n.R500, n.R500, vol_rate='gfe')
    #print(h.value-c.value, 2*h.plus, 2*h.minus, 2*c.plus, 2*c.minus)
    if (h.value - c.value) <= 2*h.minus or (h.value - c.value) <= 2*c.plus:
        if (h.value-c.value)>= -2*h.plus or (h.value-c.value)>= -2*c.minus:
            equilibrium_clusters.append(n)
            eq_heating_rates.append(h)
            eq_cooling_rates.append(c)

print(len(equilibrium_clusters))

44


In [14]:
with open('data/sp_equilibrium_clusters.pkl', 'wb') as f:
    pickle.dump(equilibrium_clusters, f)

In [15]:
with open('data/sp_clusters.pkl', 'wb') as f:
    pickle.dump(nfws_subset, f)